## I. Data Model
The core objective of the data model is to generate a continuous health risk score, where a higher score indicates higher risk. Risk levels are then assigned based on the score, and a final determination is made: an elderly person with 2 or more conditions is classified as having high medical needs. The three models perform distinct roles in a progressive framework to deliver the most reasonable prediction results.
### 1. Linear Regression (OLS) = Baseline Model
### Tasks:
•	Provide the simplest and most interpretable risk score for predicting high-risk elderly individuals
•	Dependent variable (Y): Risk count (0–9 points, counting the number of conditions present: diabetes, hypertension, hypercholesterolemia, arthritis, cardio_disease, COPD, cancer, fair/poor health, functional_problem)
•	Independent variables (X): 9 binary features (0 = absent, 1 = present)
•	Output: Predicted risk score + coefficient for each feature (positive coefficient = increased risk)
### Advantages:
•	Stable and simple model structure
•	Clearly reveals the direction and magnitude of linear effects of each risk factor on high medical needs among the elderly, and provides a reliable benchmark for comparing subsequent complex models such as Random Forest and XGBoost
### 2. Random Forest (Regression) = Feature Importance + Robust Scoring
### Tasks:
•	Automatically capture interactions between features (e.g., synergistic risk of hypertension + diabetes)
•	Output feature importance ranking (e.g., cardio_disease > cancer > COPD…)
•	Provide a moderately accurate, highly interpretable risk score to identify which health conditions most strongly affect elderly health risk
### Advantages:
•	Higher accuracy than linear regression, with risk scores that better fit real-world data
### 3. XGBoost (Regression) = Final High-Precision Scoring Model
### Tasks:
•	Fit nonlinear relationships and complex feature interactions
•	Serve as the final scoring output for matching the risk rating scale
•	Prediction target: Risk count (0–9)
•	Output: Final precise risk score (may be decimal or integer)
•	Classify risk levels based on scores → match the rating scale
### Advantages:
•	Highest accuracy among the three models
•	Delivers the most precise risk score
## II. Model Implementation
### Step 1: Data Preparation
•	X: 9 binary features (0/1)
diabetes, hypertension, hypercholesterolemia, arthritis, cardio_disease, COPD, cancer, fair/poor health, functional_problem
•	Y: Actual number of risk conditions (0–9, direct count)
### Step 2: Sequential Training & Comparison of Three Models
### 1.	OLS Linear Regression
o	Train → obtain baseline risk score
o	Evaluate: R², MAE, RMSE
### 2.	Random Forest Regression
o	Train → obtain robust score + feature importance
o	Evaluation: Must outperform linear regression
### 3.	XGBoost Regression
o	Train → obtain final high-precision score
o	Evaluation: Best performance among the three models
### Step 3: Score → Level → Business Decision
Meeting 2 or more conditions = High health risk / High medical need elderly
## III. Experimental Setup
The experiment is conducted in a standard data analysis environment with the following configuration:
•	Operating System: Windows 10/11 or macOS
•	Programming Language: Python 3.8 or above
•	Development Environment: Jupyter Notebook / PyCharm
•	Core Libraries:
o	scikit-learn: For linear regression, random forest, data preprocessing, and model evaluation
o	XGBoost: For implementing the gradient boosting model
o	Pandas & NumPy: For data cleaning and processing
o	Matplotlib & Seaborn: For result visualization and feature importance plotting
•	Data Format: CSV dataset containing demographic and health information of the elderly
•	Data Split: 80% training set, 20% test set
This is a lightweight environment that runs efficiently on an ordinary personal computer without requiring a high-performance graphics card.
## IV. Output Results
1.	Performance comparison table of the three models (demonstrating optimal performance of XGBoost)
2.	Risk factor importance ranking (output by Random Forest)
3.	Individual risk scores for elderly subjects (final output by XGBoost)
4.	List of elderly individuals classified as high health risk (≥2 conditions)
5.	Risk score-level correspondence table

## V. Preliminary Code Framework
#### ===================== 1. Import Libraries =====================
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
####  Models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

####  Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

####  ===================== 2. Define Health Features =====================
FEATURES = [
    "diabetes",
    "hypertension",
    "hypercholesterolemia",
    "arthritis",
    "cardio_disease",
    "COPD",
    "cancer",
    "fair_poor_health",
    "functional_problem"
]

#### ===================== 3. Generate Simulated Data =====================
np.random.seed(42)
n_samples = 5000

data = pd.DataFrame({
    feat: np.random.randint(0, 2, n_samples) for feat in FEATURES
})

data["true_risk_count"] = data[FEATURES].sum(axis=1)
data["true_high_risk"] = (data["true_risk_count"] >= 2).astype(int)

print("Data generation completed")
print(f"Total samples: {len(data)}")
print(f"True high-risk proportion: {data['true_high_risk'].mean():.1%}")
print(data.head())

####  ===================== 4. Train/Test Split =====================
X = data[FEATURES]
y = data["true_risk_count"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

####  ===================== 5. Train Three Models =====================
####  OLS (Baseline model)
ols = LinearRegression()
ols.fit(X_train, y_train)

#### Random Forest Regression
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

####  XGBoost Regression
xgb = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)

print("\nAll models trained successfully!")

####  ===================== 6. Model Prediction =====================
y_pred_ols = ols.predict(X_test)
y_pred_rf = rf.predict(X_test)
y_pred_xgb = xgb.predict(X_test)

####  ===================== 7. Model Evaluation =====================
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {
        "Model": model_name,
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3),
        "R²": round(r2, 3)
    }

eval_df = pd.DataFrame([
    evaluate_model(y_test, y_pred_ols, "OLS Regression"),
    evaluate_model(y_test, y_pred_rf, "Random Forest"),
    evaluate_model(y_test, y_pred_xgb, "XGBoost")
])

print("\n" + "="*50)
print("Model Performance Comparison")
print("="*50)
print(eval_df)

####  ===================== 8. Feature Importance (Random Forest) =====================
print("\n" + "="*50)
print("Feature Importance Ranking")
print("="*50)

feature_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

print(feature_importance)

#### Plot Feature Importance
plt.figure(figsize=(10, 5))
sns.barplot(x="Importance", y="Feature", data=feature_importance)
plt.title("Health Risk Feature Importance", fontsize=14)
plt.tight_layout()
plt.show()

####  ===================== 9. Final Results: XGBoost Risk Score + High-Risk Classification =====================
print("\n" + "="*50)
print("Final High-Risk Classification (XGBoost Model)")
print("Rule: Predicted risk score ≥ 2 = High health risk / High medical need")
print("="*50)

result = pd.DataFrame({
    "Actual Risk Count": y_test,
    "XGBoost Predicted Score": np.round(y_pred_xgb, 2),
    "Predicted High Risk": (y_pred_xgb >= 2).map({True: "Yes", False: "No"})
})

high_risk_num = (result["Predicted High Risk"] == "Yes").sum()
total_num = len(result)
high_risk_pct = high_risk_num / total_num

print(f"Total in test set: {total_num}")
print(f"Predicted high-risk individuals: {high_risk_num}")
print(f"High-risk proportion: {high_risk_pct:.1%}")
print("\nTop 10 results:")
print(result.head(10))

#### ===================== 10. Model Comparison Visualization =====================
plt.figure(figsize=(8, 5))
sns.barplot(x="Model", y="R²", data=eval_df, palette="viridis")
plt.title("Model R² Comparison (Higher = Better)", fontsize=14)
plt.ylim(0, 1)
plt.show()

print("\nExecution completed!")
print("\nSummary:")
print("- OLS: Baseline model")
print("- Random Forest: Feature importance analysis")
print("- XGBoost: Final risk prediction")
print("- Rule: Score ≥ 2 = High-risk elderly")
